# SIDED Dataset Exploration

This notebook explores the SIDED (Synthetic Industrial Dataset for Energy Disaggregation) dataset.

## Overview

The SIDED dataset contains synthetic time series data for industrial facilities with 5 appliance types:
- **EVSE**: Electric Vehicle Supply Equipment
- **PV**: Photovoltaic (solar generation)
- **CS**: Cooling Systems
- **CHP**: Combined Heat and Power
- **BA**: Base Appliances

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

from edge_pipeline.data.sided_loader import SidedLoader, APPLIANCE_ORDER

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

## Load Data

In [ ]:
loader = SidedLoader()
facilities = loader.load_all()

print(f"Loaded {len(facilities)} facilities")
for f in facilities:
    print(f"  {f.facility_id}: {len(f)} samples ({f.num_days} days)")

## Visualise Aggregate and Appliances

In [ ]:
# Plot first facility, first day
facility = facilities[0]
samples_per_day = 24 * 60 // facility.resolution_minutes

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Aggregate power
axes[0].plot(facility.aggregate[:samples_per_day], label="Aggregate", color="black", linewidth=1.5)
axes[0].set_ylabel("Power (W)")
axes[0].set_title(f"Facility {facility.facility_id} - Day 1")
axes[0].legend()

# Individual appliances
colors = ["#E41A1C", "#377EB8", "#4DAF4A", "#984EA3", "#FF7F00"]
for i, (name, color) in enumerate(zip(APPLIANCE_ORDER, colors)):
    axes[1].plot(facility.appliances[:samples_per_day, i], label=name, color=color, alpha=0.8)

axes[1].set_xlabel("Time (samples)")
axes[1].set_ylabel("Power (W)")
axes[1].legend(ncol=5, loc="upper right")

plt.tight_layout()
plt.show()

## Power Distribution by Appliance

In [ ]:
# Combine all facility data
all_appliances = np.concatenate([f.appliances for f in facilities])

fig, axes = plt.subplots(1, 5, figsize=(15, 3))

for i, (name, ax) in enumerate(zip(APPLIANCE_ORDER, axes)):
    data = all_appliances[:, i]
    ax.hist(data, bins=50, edgecolor="black", alpha=0.7)
    ax.set_title(name)
    ax.set_xlabel("Power (W)")
    ax.axvline(np.mean(data), color="red", linestyle="--", label=f"Mean: {np.mean(data):.0f}")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## Relative Contribution Analysis

This analysis is important for AMDA augmentation, which scales appliances inversely to their contribution.

In [ ]:
# Calculate relative contributions
total_abs_power = np.sum(np.abs(all_appliances), axis=1, keepdims=True)
relative_contributions = np.abs(all_appliances) / (total_abs_power + 1e-8)

mean_contributions = relative_contributions.mean(axis=0)

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(APPLIANCE_ORDER, mean_contributions, color=colors, edgecolor="black")
ax.set_ylabel("Mean Relative Contribution")
ax.set_title("Average Relative Power Contribution by Appliance Type")

for bar, contrib in zip(bars, mean_contributions):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
            f"{contrib:.1%}", ha="center", fontsize=10)

plt.tight_layout()
plt.show()

## Correlation Analysis

In [ ]:
import pandas as pd

df = pd.DataFrame(all_appliances, columns=APPLIANCE_ORDER)
df["Aggregate"] = np.sum(all_appliances, axis=1)

corr_matrix = df.corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap="RdBu_r", center=0, 
            vmin=-1, vmax=1, ax=ax, fmt=".2f")
ax.set_title("Correlation Matrix: Appliances and Aggregate")
plt.tight_layout()
plt.show()